# Figure 2: What carries cross-species homology

Figure 1 left one thing unexplained. The evidence tiers grade *resolution* rather than existence.
Anchored and non-anchored parcels find the human region equally well (AUROC 0.87 vs 0.88), but only
anchored parcels find the right **parcel** (top-1 0.69 vs 0.18).

We asked why. The answer:

> **Connectivity and spatial position carry *which human region* a mouse region corresponds to.
> Curation carries *which parcel*.**

Three lines of evidence:

1. **Ablation.** Strip the model's cost terms one at a time and watch which metric moves (Fig. 2a).
2. **Leave-one-region-out.** Withhold each curated unit, re-fit, and score its recovery from
   connectivity and space alone (Fig. 2c).
3. **An external benchmark the model never saw.** Beauchamp's 19 transcriptomically-derived homology
   pairs (Fig. 2b, d, e).

HOMER is therefore neither a connectivity-only method nor a landmark look-up. Connectivity on its own is
unidentifiable, because Gromov–Wasserstein aligns two connectomes only up to relabelling. Both halves of
the objective do work.

> ⚠️ **Panel letters.** In the composed figure the order is **a** = ablation, **b** = Beauchamp brains,
> **c** = held-out LORO, **d** = displacement, **e** = curation-removed. An earlier draft of the text had
> **b** and **c** swapped. The order below follows the composed figure.
>
> ⚠️ **41 units = 15 Garin classes + 26 packs.** An earlier draft said "21 Garin classes", which does not
> add up (21 + 26 = 47). Only 15 of the 21 Garin classes were testable in the leave-one-out.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
LOGS = ROOT / 'outputs' / 'logs'

# These experiments each re-fit the FGW model dozens of times (the LORO alone re-fits 41 times, the
# weight scan 125 times). That is hours of compute, so the notebook reads their persisted results
# rather than re-running them. The scripts that produce them:
#   Fig 2a  manuscript/figures/fig_2_ED/ablation_ladder.py       -> ablation_ladder_battery.json
#   Fig 2c  manuscript/figures/fig_2_ED/leave_one_region_out.py  -> anchor_recovery_loo_combined.json
#   Fig 2b,d manuscript/figures/fig_2_ED/beauchamp_battery.py    -> beauchamp_metric_battery.json
#   Fig 2e  manuscript/figures/fig_2_ED/disentangle_loro.py      -> beauchamp_metric_battery_loro.json
lad  = json.loads((LOGS / 'ablation_ladder_battery.json').read_text())
loo  = json.loads((LOGS / 'anchor_recovery_loo_combined.json').read_text())
bea  = json.loads((LOGS / 'beauchamp_metric_battery.json').read_text())
loro = json.loads((LOGS / 'beauchamp_metric_battery_loro.json').read_text())
abl  = json.loads((LOGS / 'ablation_auroc.json').read_text())

print('ablation ladder rungs :', list(lad))
print('LORO units            :', len(loo))
print('Beauchamp regions     :', len(bea['per_region']))

## 1. Decomposition: strip the cost terms one at a time (Fig. 2a)

HOMER's objective has two halves:

- a **relational (Gromov–Wasserstein)** term on the two connectomes, which asks whether two parcels
  relate to their neighbours in the same way;
- a **cross-species feature** term, which is normalised spatial position plus the curated homology
  constraints.

The ladder adds them in turn. We read two metrics side by side:

- **AUROC**, region-level recovery: does the mouse region's mass concentrate on the *right human region*?
- **top-1 and displacement**, parcel-exact recovery: does it land on the *right parcel*?

**Connectivity alone recovers homologues at chance** (AUROC 0.67, top-1 0.8 %). This is a mathematical
property of the objective rather than a failure of the data. Gromov–Wasserstein is invariant to
relabelling, so with nothing to fix the global orientation the coupling cannot be *placed*. The
connectomes carry the shape of the relationship without carrying where to put it.

In [ ]:
rungs = ['connectivity', '+spatial', '+anchors', '+packs']
rungs = [r for r in rungs if r in lad]
# DISPLACEMENT: two different quantities live in these logs and they differ by 2x.
#   centroid_disp_mm: distance from the CENTROID of the routed mass to the true region. <- the
#                     manuscript's number, and what make_beauchamp_report.py plots.
#   expected_disp_mm: the mass-weighted EXPECTED distance, which a broad distribution inflates.
# Both are legitimate; they answer different questions. Fig. 2a and 2d report the centroid.
DISP = 'centroid_disp_mm'
rows = [(r, lad[r]['auroc'], lad[r]['top1'], lad[r][DISP]) for r in rungs]

print(f"{'cost terms':<16} {'AUROC':>7} {'top-1':>8} {'displacement':>14}")
print('-' * 50)
for name, a, t, d in rows:
    print(f'{name:<16} {a:>7.2f} {t:>7.1%} {d:>11.0f} mm')
print()
print('AUROC saturates as soon as spatial position is added and is then UNCHANGED by curation.')
print('top-1 and displacement move only when the anchors and packs arrive.')
print('The two quantities have two different sources.')

In [ ]:
# ---------------- Fig 2a: the decomposition ladder ----------------
x = np.arange(len(rows))
auroc = [r[1] for r in rows]
top1 = [r[2] for r in rows]
disp = [r[3] for r in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(x, auroc, 'o-', color='#1b4f8a', lw=2.4, ms=9, label='region-level (AUROC)', zorder=3)
ax.axhline(0.5, color='0.7', ls=':', lw=1)
ax.text(-0.35, 0.51, 'chance', fontsize=8, color='0.5')
ax.set_ylabel('region-level recovery (AUROC)', color='#1b4f8a')
ax.set_ylim(0.4, 1.0)
ax.tick_params(axis='y', labelcolor='#1b4f8a')
ax.set_xticks(x)
ax.set_xticklabels(['connectivity\n(GW on FC+SC)', '+ spatial\nposition', '+ curated\nanchors',
                    '+ region\npacks'][:len(rows)], fontsize=9)

ax2 = ax.twinx()
ax2.plot(x, top1, 's--', color='#e08a2b', lw=2.2, ms=8, label='parcel-exact (top-1)', zorder=3)
ax2.set_ylabel('parcel-exact recovery (top-1)', color='#e08a2b')
ax2.set_ylim(0, 0.6)
ax2.tick_params(axis='y', labelcolor='#e08a2b')
ax2.spines['top'].set_visible(False)
for xi, d in zip(x, disp):
    ax2.annotate(f'{d:.0f} mm', (xi, top1[int(xi)]), textcoords='offset points', xytext=(0, -16),
                 ha='center', fontsize=8, color='#8a5a12')

ax.spines['top'].set_visible(False)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, fontsize=9, loc='center right')
ax.set_title('Connectivity + space carry WHICH REGION; curation carries WHICH PARCEL\n'
             f'AUROC {auroc[0]:.2f} → {auroc[1]:.2f} → {auroc[-1]:.2f}   '
             f'top-1 {top1[0]:.0%} → {top1[1]:.0%} → {top1[-1]:.0%}',
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

### The factorial control (ED2a,b)

The ladder tests one path through the model. The factorial tests every cell, and one cell is the sharp
control: **gene-coexpression connectivity alone lands *below chance* (AUROC 0.28).** A connectivity
matrix, however biologically meaningful, cannot place itself without a cross-species reference.

In [ ]:
print('cross-species term varied, relational cost held at FC + SC:')
for k in ('GW_only_(no_M)', 'xyz_only', 'xyz_anchors', 'xyz_anch_M_gene'):
    if k in abl:
        print(f'  {k:20s} AUROC {abl[k]:.2f}')
print()
print('relational modality alone, no cross-species reference at all:')
for k in ('grid_FC_none', 'grid_SC_none', 'grid_gene_none'):
    if k in abl:
        print(f'  {k:20s} AUROC {abl[k]:.2f}')
print()
print('gene-coexpression connectivity ALONE falls BELOW chance (0.5). Connectivity is unidentifiable')
print('without a cross-species reference. Spatial position is the single largest lift in the model.')

## 2. Withhold the curation entirely (Fig. 2c)

The obvious objection to §1 is that the coupling might be echoing back the regions we anchored.

So we remove them. For each of the **41** combined supervision units (15 Garin homology classes + 26
region packs), we delete that unit's anchors, **re-fit the full model**, and score recovery of the
held-out unit from connectivity and space alone.

In [ ]:
kinds = {}
for name, d in loo.items():
    kinds.setdefault(d.get('kind', 'unknown'), []).append(d)

def wmean(ds, key):
    """Parcel-count-weighted mean. The manuscript's 0.73/0.74/0.72 are WEIGHTED means."""
    w = np.array([d['n_mouse'] for d in ds], float)
    v = np.array([d[key] for d in ds], float)
    return float((w * v).sum() / w.sum())

allds = [d for ds in kinds.values() for d in ds]
print(f'{len(allds)} held-out units:')
for k, ds in sorted(kinds.items()):
    print(f'  {k:8s} n = {len(ds):>2d}   held-out AUROC {wmean(ds, "auroc"):.2f}   '
          f'top-1 {wmean(ds, "top1"):.3f}')
print(f'  {"OVERALL":8s} n = {len(allds):>2d}   held-out AUROC {wmean(allds, "auroc"):.2f}   '
      f'top-1 {wmean(allds, "top1"):.3f}')
print()
print('Region-level recovery holds WELL above chance (0.5). Parcel-exact recovery COLLAPSES to ~2 %.')
print()
print('NOTE: these are PARCEL-COUNT-WEIGHTED means. The unweighted mean AUROC is'
      f' {np.mean([d["auroc"] for d in allds]):.3f}.')
print('The manuscript reports the weighted values (0.73 / 0.74 / 0.72). During the audit these were')
print('briefly "corrected" to the unweighted number. They were right. Match the definition before')
print('changing a number.')

In [ ]:
# ---------------- Fig 2c: per-unit held-out AUROC ----------------
garin = sorted(kinds.get('garin', []), key=lambda d: -d['auroc'])
packs = sorted(kinds.get('pack', []), key=lambda d: -d['auroc'])
vals = [d['auroc'] for d in garin] + [d['auroc'] for d in packs]
cols = ['#1b4f8a'] * len(garin) + ['#e08a2b'] * len(packs)

fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.scatter(range(len(vals)), vals, c=cols, s=42, zorder=3, edgecolor='white', linewidth=0.6)
ax.axhline(0.5, color='0.6', ls='--', lw=1)
ax.text(len(vals) - 0.5, 0.505, 'chance', fontsize=8, color='0.5', ha='right')
full = lad[rungs[-1]]['auroc']
held = wmean(allds, 'auroc')
ax.axhline(full, color='#333', lw=1.6, label=f'full model ({full:.2f})')
ax.axhline(held, color='#333', ls=':', lw=1.6, label=f'held-out mean ({held:.2f})')
ax.set_ylabel('held-out region-level recovery (AUROC)')
ax.set_xlabel('supervision unit (each removed in turn, model re-fit)')
ax.set_ylim(0, 1.02)
ax.set_xticks([])
from matplotlib.lines import Line2D
handles, labels = ax.get_legend_handles_labels()
handles += [Line2D([], [], marker='o', ls='', color='#1b4f8a', label=f'Garin class (n = {len(garin)})'),
            Line2D([], [], marker='o', ls='', color='#e08a2b', label=f'region pack (n = {len(packs)})')]
ax.legend(handles=handles, frameon=False, fontsize=8.5, ncol=2, loc='lower left')
ax.set_title('Withholding an anchor costs parcel precision, not region-level correspondence\n'
             f'held-out AUROC {held:.2f} (chance 0.5), while held-out top-1 collapses to '
             f'{wmean(allds, "top1"):.0%}',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

### What leave-one-region-out shows

Held-out top-1 is ~2 %. Reporting that as "HOMER localises 45.7 % → 2 % without anchors" would be
misleading.

**Top-1 is the wrong metric for a held-out unit.** When a region's own anchors are gone, the coupling
still routes its mass to roughly the right *place*. It cannot pick the exact parcel out of ~2,000.
Scored by **millimetre displacement** rather than top-1, connectivity and space give coarse but real
localisation. This is the AUROC-versus-top-1 dissociation seen from the other side.

In [ ]:
d_full = np.array([d[DISP] for d in allds], float)
print('held-out centroid displacement from the true human region:')
print(f'  median {np.median(d_full):.0f} mm   IQR {np.percentile(d_full, 25):.0f}–'
      f'{np.percentile(d_full, 75):.0f} mm')
print()
print('Compare with the full model in the ladder above: '
      f"{lad[rungs[-1]][DISP]:.0f} mm.")
print()
print('So the held-out coupling is displaced, but it is not lost. Connectivity and space give coarse')
print('localisation; the anchors sharpen it to the parcel. Reporting the top-1 collapse ALONE would')
print('overstate the failure.')

## 3. An external benchmark the model never saw (Fig. 2b, d, e)

Beauchamp et al. derive 19 mouse–human region correspondences from **whole-brain transcriptomic
similarity**, a modality HOMER does not use, in a benchmark that never entered the fit. We scored them
against the frozen production coupling.

In [ ]:
per = bea['per_region']
names = list(per)
auroc_b = np.array([per[n]['auroc'] for n in names])
mass_b  = np.array([per[n]['mass_in_region'] for n in names])
disp_b  = np.array([per[n][DISP] for n in names])
w       = np.array([per[n]['n_mouse'] for n in names], float)
q       = np.array([per[n]['perm_q_mass'] for n in names])

wm = lambda v: float((w * v).sum() / w.sum())
n_sig = int((q < 0.05).sum())

print(f'{len(names)} Beauchamp pairs, scored on the frozen production coupling:')
print(f'  region-level AUROC   {wm(auroc_b):.2f}')
print(f'  mass-in-region       {wm(mass_b):.2f}')
print(f'  displacement         {wm(disp_b):.0f} mm  (parcel-weighted mean)')
print(f'  significant          {n_sig} / {len(names)} regions '
      f'(parcel-set permutation null, FDR q < 0.05)')
print()
print('The errors are graceful. A miss lands on an anatomically ADJACENT structure rather than')
print('scattering across the brain. The displacement metric detects that; top-1 does not.')

In [ ]:
# ---------------- Fig 2d: displacement per region, coloured by AUROC ----------------
order = np.argsort(disp_b)
fig, ax = plt.subplots(figsize=(7.4, 5.6))
sc = ax.barh(range(len(names)), disp_b[order],
             color=plt.cm.viridis((auroc_b[order] - 0.5) / 0.5))
ax.axvline(wm(disp_b), color='#c1272d', ls='--', lw=1.6,
           label=f'parcel-weighted mean ({wm(disp_b):.0f} mm)')
ax.set_yticks(range(len(names)))
ax.set_yticklabels([names[i].split(' -> ')[0] for i in order], fontsize=7.5)
ax.set_xlabel('displacement of routed mass from the expected homologue (mm)')
ax.legend(frameon=False, fontsize=9)
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0.5, 1.0))
cb = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.02); cb.set_label('region-level AUROC', fontsize=9)
ax.set_title('Errors are graceful: a miss lands on an adjacent structure\n'
             f'AUROC {wm(auroc_b):.2f}, mass-in-region {wm(mass_b):.2f}, '
             f'{n_sig}/{len(names)} significant (FDR q < 0.05)',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

### The memorisation control (Fig. 2e)

Several region packs were curated **on Beauchamp regions**, so the benchmark is not fully independent
until we remove that overlap. We delete each region's overlapping curation, **re-fit the model**, and
re-score.

In [ ]:
common = [n for n in names if n in loro]
a_full = np.array([bea['per_region'][n]['auroc'] for n in common])
a_held = np.array([loro[n]['auroc'] for n in common])
wc = np.array([bea['per_region'][n]['n_mouse'] for n in common], float)
agg_full = float((wc * a_full).sum() / wc.sum())
agg_held = float((wc * a_held).sum() / wc.sum())

print(f'{len(common)} regions re-scored with their overlapping curation removed and the model re-fit:')
print(f'  aggregate AUROC   {agg_full:.2f}  ->  {agg_held:.2f}')
print(f'  regions that drop below chance: {(a_held < 0.5).sum()} of {len(common)}')
print()
print('Recovery is largely retained. Agreement with the transcriptomic benchmark therefore reflects')
print('cross-species signal that the coupling RECONSTRUCTS rather than curation it MEMORISED.')

fig, ax = plt.subplots(figsize=(5.4, 5.2))
ax.scatter(a_full, a_held, s=46, c='#1b4f8a', alpha=0.8, edgecolor='white', linewidth=0.6, zorder=3)
lim = [0.3, 1.02]
ax.plot(lim, lim, color='0.6', ls='--', lw=1, label='no loss')
ax.axhline(0.5, color='0.85', lw=1); ax.axvline(0.5, color='0.85', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('full-model AUROC')
ax.set_ylabel('AUROC with overlapping curation removed')
ax.legend(frameon=False, fontsize=9, loc='lower right')
ax.set_title('Agreement is not memorised curation\n'
             f'aggregate AUROC {agg_full:.2f} → {agg_held:.2f} when the overlap is removed',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 4. Summary

| test | region-level | parcel-exact |
|---|---|---|
| connectivity only (GW on FC + SC) | AUROC 0.67 (**chance**) | top-1 0.8 % |
| + spatial position | AUROC **0.87** (saturated) | top-1 8 % |
| + curated anchors and packs | AUROC 0.85 (**unchanged**) | top-1 **46 %**, displacement 28 → 17 mm |
| curation withheld (LORO, 41 units) | AUROC **0.73** (holds) | top-1 ≈ 2 % (collapses) |
| external benchmark (Beauchamp, 19 pairs) | AUROC 0.85, 18/19 significant | displacement 17 mm |
| benchmark with overlapping curation removed | AUROC 0.85 → **0.78** | n/a |

**Connectivity and space carry which region. Curation carries which parcel.**

This is the fact that makes the rest of the paper possible. If the coupling were a landmark look-up, §3
would be circular: you cannot test whether a *coupling* transfers connectional organisation when the
coupling is just the anchors. The region-level correspondence survives withholding every anchor, and it
agrees with an independent transcriptomic benchmark that never entered the fit.

### Panels not produced here

Fig. 2b (the 19 Beauchamp routed-mass brains) and ED2d (held-out AUROC painted on the mouse brain) are
volumetric renderings; they come from `manuscript/figures/fig_2_ED/make_beauchamp_report.py` and
`make_scan_surface.py`. ED2c (the 125-cell weight scan) reads `scan_weights.json`.